# Phase 1: Knee MRI Data Structure & Integrity Audit (Repo-Style)

This notebook runs the **Phase 1 Data Integrity Audit** following the repo-style workflow defined in `INSTRUCTIONS.md`.
All core logic lives in `src/data_audit.py` in the git repository.

### Purpose:
- Clone/pull latest repo code.
- Purge module cache to ensure the newest version runs.
- Execute `run_integrity_audit()` on competition data.
- Display distributions and confirm zero hard data errors.

In [ ]:
# --- Cell 1: Sync Repo (Kaggle Notebook Top Cell) ---
import os, sys, subprocess
from pathlib import Path

# Set your repository URL (or use Kaggle Secrets if private)
REPO_URL = os.environ.get("REPO_URL", "https://github.com/Prashanf/Rsna_KneeAbnormality_Detection.git")
REPO_DIR = "/kaggle/working/repo"

# If running inside Kaggle environment
if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo from {REPO_URL}...")
        try:
            subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Git clone failed (check REPO_URL or network): {e}")
    else:
        print("Pulling latest commits...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    if os.path.exists(REPO_DIR):
        commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip()
        print(f"Repo synced at commit: {commit}")
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
else:
    # Local environment fallback
    local_root = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    print(f"Running in local workspace: {local_root}")
    if local_root not in sys.path:
        sys.path.insert(0, local_root)

In [ ]:
# --- Cell 2: Purge module cache & load consolidated imports ---
for mod in list(sys.modules):
    if mod.startswith("src.") or mod.startswith("environment."):
        del sys.modules[mod]

from environment.imports import *
print("Consolidated environment imports loaded successfully!")

In [ ]:
# --- Cell 3: Detect paths & dataset location ---
def resolve_paths():
    kaggle_roots = list(Path('/kaggle/input').glob('*knee*')) + list(Path('/kaggle/input').glob('*rsna*'))
    if kaggle_roots:
        raw_dir = kaggle_roots[0]
        processed_dir = Path('/kaggle/working/data/processed')
        return raw_dir, processed_dir

    # Local path fallbacks
    for candidate in [Path('data/raw'), Path('../data/raw')]:
        if candidate.exists():
            return candidate.resolve(), candidate.parent.resolve() / 'processed'

    return Path('data/raw'), Path('data/processed')

RAW_DIR, PROCESSED_DIR = resolve_paths()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed artifacts directory: {PROCESSED_DIR}")

In [ ]:
# --- Cell 4: Run Data Integrity Audit from repo ---
report = run_integrity_audit(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    sample_slices=2,
)

In [ ]:
# --- Cell 5: Inspect and visualize audit distributions ---
import matplotlib.pyplot as plt

report_path = PROCESSED_DIR / 'integrity_report.json'
with open(report_path) as f:
    data = json.load(f)

dists = data.get('distributions', {})
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Series per study distribution
sps_counts = dists.get('series_per_study', {}).get('counts_distribution', {})
if sps_counts:
    axes[0, 0].bar([str(k) for k in sps_counts.keys()], list(sps_counts.values()), color='#2b5c8f')
    axes[0, 0].set_title('Series Count per Study')
    axes[0, 0].set_xlabel('Number of Series')
    axes[0, 0].set_ylabel('Number of Studies')
    axes[0, 0].grid(axis='y', linestyle='--', alpha=0.6)

# 2. Anatomical planes distribution
planes = dists.get('anatomical_planes', {})
if planes:
    axes[0, 1].bar(list(planes.keys()), list(planes.values()), color='#d95f02')
    axes[0, 1].set_title('Anatomical Planes Distribution')
    axes[0, 1].set_xlabel('Plane')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].grid(axis='y', linestyle='--', alpha=0.6)

# 3. Contrast attributes (Fluid Sensitivity / Fat Sat)
fluid = dists.get('fluid_sensitive', {})
fat = dists.get('fat_suppression', {})
categories = ['Fluid Sens (1)', 'Fluid Sens (0)', 'Fat Sat (1)', 'Fat Sat (0)']
vals = [fluid.get('1', fluid.get(1, 0)), fluid.get('0', fluid.get(0, 0)),
        fat.get('1', fat.get(1, 0)), fat.get('0', fat.get(0, 0))]
axes[1, 0].bar(categories, vals, color='#2ca02c')
axes[1, 0].set_title('Series Protocol Contrast Attributes')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(axis='y', linestyle='--', alpha=0.6)

# 4. Target label positive prevalence in train.csv
labels_prev = dists.get('label_prevalence_raw_train_csv', {})
if labels_prev:
    pos_rates = {k: v['positive_rate'] for k, v in labels_prev.items() if v['positive_rate'] is not None}
    axes[1, 1].barh(list(pos_rates.keys()), [v * 100 for v in pos_rates.values()], color='#7570b3')
    axes[1, 1].set_title('Train.csv Label Positive Prevalence (%)')
    axes[1, 1].set_xlabel('Positive Rate (%)')
    axes[1, 1].grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 6: Verify outputs generated on disk ---
parquet_path = PROCESSED_DIR / 'series_index.parquet'
if parquet_path.exists():
    df_series = pd.read_parquet(parquet_path)
    print(f"Successfully generated series index: {len(df_series)} entries")
    display(df_series.head(10))
else:
    print(f"Warning: {parquet_path} was not generated.")